---
title: Introduction to Neural Networks (Part 1)
short_title: Part 1
subject: DEEP LEARNING
---

## Introduction

Deep learning models are machine learning models with multiple layers of learned representations. A **layer** refers to any transformation that maps an input to its **feature representation**. In principle, any function that can be described as a composition of layers with *differentiable operations*[^1] is called a **neural network**. Neural networks discover intricate structure in large datasets by learning good representations ({numref}`01-thought`).

[^1]: Differentiable operations allow the network to 
change its layer parameters based on prediction errors via an application of the chain rule while tracking functional dependencies.


```{figure} ./img/01-thought.png
---
name: 01-thought
width: 80%
align: center
---
**Thought as representation.** Representations are powerful. In machine translation, sentences are not translated word-for-word between two languages. Instead, a model learns a latent representation that captures the 'thought' shared by the translated sentences. 

```

The practical successes of deep learning can be attributed to three factors:

| | |
|--|--|
| **Scalability**| Deep neural networks ({numref}`01-imagenet-progress`) are created by stacking multiple layers (10-100+). Features at deep layers can be thought of as combinatorial, higher-level, and less sensitive to noise, thereby improving generalization. On the other hand, *wider layers*, i.e. layers with more parameters, memorizes features better at that level. There are tradeoffs with depth and width that has to be controlled in order to get effectively sized networks. Generally, the larger the network, the better it generalizes to test data ({numref}`01-gflops`).|
| **Large datasets** | Deep large networks ({numref}`01-imagenet-progress`) are necessary for tasks involving massive, complex, structured datasets like images and text, where the complexity of the model aligns with that of the underlying distributions in the data. Complex models allow for *automatic feature engineering*, provided that prior knowledge about the structure (or [modality](https://en.wikipedia.org/wiki/Multimodal_learning)) of the data is encoded into the network architecture. More importantly, it turns out that knowledge that neural networks learned can be re-used via **transfer learning** where are a large model pre-trained on a large dataset is adapted to specialized tasks requiring smaller datasets.|
| **Compute** | Both of the above factors require significant computational resources ({numref}`01-gflops`). GPUs are particularly well-suited for the large scale matrix operations required in deep learning with their parallel processing capabilities. |

```{figure} ./img/01-imagenet-progress.png
---
name: 01-imagenet-progress
width: 80%
align: center
---
Progress in top-5 error in the [ImageNet competition](https://en.wikipedia.org/wiki/ImageNet#ImageNet_Challenge).
```

```{figure} ./img/01-gflops.png
---
name: 01-gflops
width: 80%
align: center
---
**[Top-1 accuracy](https://stackoverflow.com/questions/37668902/evaluation-calculate-top-n-accuracy-top-1-and-top-5) vs [GFLOPs](https://en.wikipedia.org/wiki/FLOPS) on ImageNet.** Performance generally improve with increasing compute. But some architectures such as ResNet have better tradeoff than others (e.g. VGG).
```

## Fully-connected NNs

**Fully-connected networks**[^fcnn] (FNN) consist of **linear layers** followed by an **activation**. Each layer consist of neurons that compute $\textbf{\textsf{x}}_k^{\ell + 1} = \varphi(\textbf{\textsf{x}}^{\ell} \cdot \textbf{\textsf{w}}_k^\ell + \mathsf{b}_k^\ell)$
for $k = 1, \ldots, h^\ell$ for an input $\textbf{\textsf{x}} = \textbf{\textsf{x}}^0$ where the activation $\varphi\colon \mathbb{R} \to \mathbb{R}$ is some nonlinear function. The number of neurons $h^\ell$ is called the **width** of the $\ell${sup}`th` layer. The number of layers in a network is called its **depth**. In practice, the neurons in a layer are computed in parallel using matrix operations:

$$
\underbrace{\textbf{\textsf{X}}^1}_{{B \times h}} = \varphi\left(\underbrace{\textbf{\textsf{X}}^0}_{{B \times d}}\,\underbrace{\textbf{\textsf{W}}}_{d \times h} + \underbrace{\textbf{\textsf{b}}}_{1 \times h}\right)
$$

The layer output $\textbf{\textsf{X}}^1$ is then passed as input to the next layer.

[^fcnn]: Fully-connected networks are also known in the literature as [multilayer perceptrons](https://en.wikipedia.org/wiki/Perceptron) (MLPs).

```{figure} ./img/artificial-neuron.png
---
name: artificial-neuron
width: 80%
align: center
---
**Artificial neuron as simplistic mathematical model for the biological neuron.** Biological neurons are believed to remain inactive until the net input to the cell body (soma) reaches a certain threshold, at which point the neuron gets *activated* and fires an electro-chemical signal. [Source](https://jermwatt.github.io/machine_learning_refined/notes/13_Multilayer_perceptrons/13_2_Multi_layer_perceptrons.html) 
```

**Universal approximation.** It turns out that any continuous map $f\colon K \subset \mathbb{R}^d \to \mathbb{R}^m$ defined on a compact set $K$ can be approximated by a FNN {cite:p}`Cybenko1989`. Continuity on a compact domain are reasonable assumptions about a ground truth function that we assume exists. The following demo shows a one-dimensional curve approximated with a FNN having [ReLU activations](https://en.wikipedia.org/wiki/Rectifier_(neural_networks)) $\varphi(z) = \max(0, z)$ for $z \in \mathbb{R}.$

In [1]:
import torch

# Ground truth
x = torch.linspace(-2 * torch.pi, 2 * torch.pi, 1000)
y = torch.sin(x) + 0.3 * x

# Get sorted sample. Shifted for demo
B = sorted(torch.randint(30, 970, size=(24,)))
xs = x[B,]
ys = y[B,]

# ReLU approximation
z = torch.zeros(1000,) + ys[0]
for i in range(len(xs) - 1):
    if torch.isclose(xs[i + 1], xs[i]):
        m = torch.tensor(0.0)
    else:
        M = (ys[i+1] - ys[i]) / (xs[i+1] - xs[i])
        s, m = torch.sign(M), torch.abs(M)
    z += s * (torch.relu(m * (x - xs[i])) - torch.relu(m * (x - xs[i+1])))

**NOTE:** This only works for target $f$ with compact domain $[a, b]$ consistent with the theorem.

In [2]:
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

# Plotting
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(xs, ys, facecolor="none", s=12, edgecolor="k", zorder=3, label="data")
ax[0].plot(x, y, color="C1", label="f")
ax[0].set_xlabel("x")
ax[0].set_ylabel("y")
ax[0].legend();

ax[1].scatter(xs, ys, facecolor="none", s=12, edgecolor="k", zorder=4, label="data")
ax[1].plot(x, z, color="C0", label=f"relu approx. (B={len(B)})", zorder=3)
ax[1].plot(x, y, color="C1")
ax[1].set_xlabel("x")
ax[1].set_ylabel("y")
ax[1].legend()

plt.savefig("plots/00-relu_approx.svg", dpi=300, bbox_inches="tight")
plt.close(fig);

:::{figure} ./plots/00-relu_approx.svg
---
name: plots/00-relu_approx
align: center
---
**Approximating a sinusoidal function using ReLU neurons.** The algorithm works by constructing step-like functions `__/‾‾` and `‾‾\__`  but with slope $\nu$ between adjacent points by canceling out two ReLU curves with slopes +$\nu$ and -$\nu$. These step functions increment each other starting with `ys[0]` and ends with `ys[-1]`. This is an example of universal approximation done by means of increasing layer width.
:::

## Linear classification  

Neural networks classify data by learning separating hyperplanes, obtained through a sequence of transformations on the input. The class scores (called **logits**) are proportional to the distance of a data point from each hyperplane. The scores can be converted to a probability distribution over the class labels by applying a positive increasing function $f\colon \mathbb{R} \to \mathbb{R}^+$ to the logits, converting it into a probability vector:

$$
\textsf{p}_j = \frac{f(\textsf{s}_j)}{\sum_k f(\textsf{s}_k)}.
$$

A natural choice for $ f $ is the exponential function $\exp$ which maps $ \mathbb{R} $ to $ \mathbb{R}^+ $ 1-1, s.t. $ \exp(-\infty) = 0 $ and $ \exp (+\infty) = +\infty.$ Hence, $\textbf{\textsf{s}} = [-\infty, 1, H]$ becomes vector $\textbf{\textsf{p}} = [0, \epsilon, 1 - \epsilon]$ where $H \gg 1.$ This leads to the **softmax function**[^3]:  

$$
\text{Softmax}(\textbf{\textsf{s}})_j = \frac{\exp(\textsf{s}_j - \textsf{s}^*)}{\sum_k \exp(\textsf{s}_k - \textsf{s}^*)}, \quad \text{where} \;\; \textsf{s}^* = \max_j \textsf{s}_j.
$$

This formulation ensures numerical stability by subtracting the largest logit $ \textsf{s}^* $, so that the denominator is always at least 1.

**Example: Logistic Regression.** For binary classification, only one probability needs to be computed, since $ p_0 + p_1 = 1 $. In this case, softmax simplifies to the **sigmoid function**:  

$$
\textsf{p}_1 = \frac{1}{1 + \exp(-(\textsf{s}_1 - \textsf{s}_0))}
$$

where $ \textsf{p}_1  $ represents the probability of the positive class. Notice that logistic regression is equivalent to a single-layer neural network whose output are logits $ \textsf{s}_k = \textbf{\textsf{w}}_k \cdot \textbf{\textsf{x}} + \textsf{b}_k $ for $k = 1, 2.$ This allows us to rewrite the difference of logits as:  

$$
\textsf{s}_1 - \textsf{s}_0 = (\textbf{\textsf{w}}_1 - \textbf{\textsf{w}}_0) \cdot \textbf{\textsf{x}} + (\textsf{b}_1 - \textsf{b}_0),
$$

which corresponds to a single separating hyperplane.  

[^3]: A more precise name might be soft-*arg*max, as exponentiation amplifies the effect of the largest logit.

In [3]:
import numpy as np

s = np.linspace(-10, 10, 300)
p = 1 / (1 + np.exp(-s))

p_inv = lambda p: np.log(p / (1 - p))
threshold = 0.15
s_lo = p_inv(threshold)
s_hi = p_inv(1 - threshold)

# scores of positive and negative samples
s0 = [-8.0, -7.8, -6.5, -5.0, -2.5, -1.25, 0.3, -7.6]   # y = 0
s1 = [-2.2,  0.0,  6.0,  1.3,  5.0,   7.5, 7.6,  8.3]   # y = 1

In [4]:
plt.figure(figsize=(8, 4))
plt.title(r"logistic regression ($\tau = 0.15$)")
plt.plot(s, p, label=r"$p(y=1 \mid {\mathbf{x}}) = 1 / (1 + e^{-s})$", color="C1")
for i in range(len(s0)):
    plt.scatter(s0[i], 0.0, color="C0", zorder=3, edgecolor="k")
    plt.scatter(s1[i], 0.0, color="C1", zorder=3, edgecolor="k")

plt.axvspan(-10.0, s_lo, -2.0, 2.0, color="C0", label=r"$\hat{y} = 0$", alpha=0.3)
plt.axvspan( s_hi, 10.0, -2.0, 2.0, color="C1", label=r"$\hat{y} = 1$", alpha=0.3)
plt.axvspan( s_lo, s_hi, -2.0, 2.0, color="lightgray", label="???")
plt.xlabel("$s$")
plt.ylabel("$p$")
plt.legend(fontsize=10)
plt.ylim(-0.1, 1.1)
plt.grid(linestyle="dotted");

plt.savefig("plots/00-logistic-regression.svg", dpi=300, bbox_inches="tight")
plt.close();

```{figure} plots/00-logistic-regression.svg
---
label: plots/00-logistic-regression
---
**Logistic regression assigns a probability based on the sigmoid function.**
Sigmoid assigns $\textsf{p} =\frac{1}{2}$ at the hyperplane where $\textsf{s} = 0.$ The probability scales symmetrically away from the decision boundary in both directions. This is nice, otherwise the prediction will not be invariant with respect to relabeling. Note that this example shows a model with well-calibrated scores.
```


For the general case of multiclass classification, we have class scores $\textsf{s}_k = \textbf{\textsf{w}}_k \cdot \textbf{\textsf{x}} + \textsf{b}_k$ with softmax acting like a **voting function**[^softmax]. The weight vector $\textbf{\textsf{w}}_k$ can be interpreted as a learned pattern, so that the projection $\textbf{\textsf{x}} \cdot \textbf{\textsf{w}}_k$ corresponds to a scaled similarity. This degree of similarity is then modulated by the activation $\varphi,$ with the bias $\textsf{b}_k$ shifting the activation threshold.

[^softmax]: [This video](https://www.youtube.com/watch?v=p-6wUOXaVqs) covers further reasons why the softmax function is good for constructing probabilities.

In [5]:
# 2d grid of points
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x, y)

# parameters
w = [1, 1]
b = 0

def sigmoid_neuron(x0, x1):
    z = w[0] * x0 + w[1] * x1 + b
    return 1 / (1 + np.exp(-z))

def relu_neuron(x0, x1):
    z = w[0] * x0 + w[1] * x1 + b
    return np.maximum(0, z)

In [6]:
from matplotlib.colors import LinearSegmentedColormap

colors = ["C0", "C1"]
n_bins = 100
cm = LinearSegmentedColormap.from_list(name="", colors=colors, N=n_bins)

def plot_grid(ax, X, Y, Z, title=""):
    p1 = (-0.45, -0.45)
    p2 = (1, 1)
    ax.annotate(r'$\mathsf{w}$', xy=p2, xytext=p1, arrowprops=dict(arrowstyle='->'))
    ax.plot(x, -x, color='k', linewidth=1, linestyle='dotted', label='H')

    im = ax.pcolormesh(X, Y, Z, shading='auto', cmap=cm, rasterized=True)
    _ = plt.colorbar(im, ax=ax)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(title, size=12)
    ax.legend()


fig, ax = plt.subplots(1, 2, figsize=(11, 4))
Z0 = sigmoid_neuron(X, Y)
Z1 = relu_neuron(X, Y)
plot_grid(ax[0], X, Y, Z0, title="sigmoid")
plot_grid(ax[1], X, Y, Z1, title="ReLU")
plt.savefig("plots/00-sigmoid-relu.svg", dpi=300, bbox_inches="tight")
plt.close(fig);

```{figure} plots/00-sigmoid-relu.svg
---
label: plots/00-sigmoid-relu
---
**Activation for the input space given a weight vector $\textbf{\textsf{w}}$ and zero bias.**
The sigmoid unit assigns a value gradient along the normal of the hyperplane defined by $\textbf{\textsf{w}}$ and $\textsf{b}$ in the input space $\mathbb{R}^2$. This effectively reduces the input
space to one dimension along the direction of ${\textbf{\textsf{w}}} = [1, 1].$ Here $H$ is the linear decision boundary defined by points $\textbf{\textsf{x}}$ such that $\textbf{\textsf{x}} \cdot \textbf{\textsf{w}} + \textsf{b} = 0.$ ReLU unit has sharp cutoff at zero.
```

## Loss functions

Neural network training follows four steps: (1) defining a model, (2) defining a loss function, (3) choosing an optimizer, and (4) running it on compute (e.g. GPUs). A **loss function** acts a smooth surrogate to the true objective which may not be amenable to available optimization 
techniques. Hence, we can think of loss functions as a measure of model quality.
The choice of loss function determines what the model parameters will optimize towards.

```{figure} ./img/02-loss-surface.png
---
name: 01c-loss-surface
width: 60%
align: center
---
Loss surface for a model with two weights. [Source](https://cs182sp21.github.io/static/slides/lec-4.pdf)
```

### MLE = NLL loss

Here we derive a loss function based on the principle of [maximum likelihood estimation](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation) (MLE), i.e. finding optimal parameters $\hat{\boldsymbol{\Theta}}$ such that the dataset is assigned the highest probability under $\hat{\boldsymbol{\Theta}}.$ Consider a parametric model of the target denoted by $p_{\boldsymbol{\Theta}}(\textsf{y} \mid \boldsymbol{\mathsf{x}}).$ 
The **likelihood** of the [IID](https://en.wikipedia.org/wiki/Independent_and_identically_distributed_random_variables) sample $\mathcal{D} = \{(\boldsymbol{\mathsf{x}}_i, \textsf{y}_i)\}_{i=1}^N$ can be defined as

$$
\begin{aligned}
{L}(\boldsymbol{\Theta}) 
= \left({\prod_{i=1}^N {p_{\boldsymbol{\Theta}}(\textsf{y}_i \mid \boldsymbol{\mathsf{x}}_i)}}\right)^{\frac{1}{N}}.
\end{aligned}
$$

This is proportional to the probability assigned by the parametric model with parameters $\boldsymbol{\Theta}$ on the sample $\mathcal{D}.$
The IID assumption is important. Note that maximizing the likelihood results in a model that focuses more on inputs that are more probable since they are better represented in the sample. 
Probabilities are
small numbers in $[0, 1]$, so applying the logarithm to convert the large product to a sum
is a good idea:

$$
\begin{aligned}
\log {L}(\boldsymbol{\Theta}) 
&= \frac{1}{N}\sum_{i=1}^N \log p_{\boldsymbol{\Theta}}(\textsf{y}_i \mid \boldsymbol{\mathsf{x}}_i).
\end{aligned}
$$

MLE then maximizes the log-likelihood with respect to the parameters $\boldsymbol{\Theta}.$ The idea is that a good model makes training data more probable. It is customary in machine learning to convert this to a minimization problem. The following then becomes our optimization problem:

$$\hat{\boldsymbol{\Theta}} = \underset{\boldsymbol{\Theta}}{\text{argmin}}\,\left( -\frac{1}{N}\sum_{i=1}^N \log p_{\boldsymbol{\Theta}}(\textsf{y}_i \mid \boldsymbol{\mathsf{x}}_i)\right).$$

This allows us to define $\ell = -\log p_{\boldsymbol{\Theta}}(\textsf{y} \mid \boldsymbol{\mathsf{x}}).$ In general, the loss function can be any nonnegative function whose value approaches zero whenever the prediction of the network the target value. Observe that:

- $p_{\boldsymbol{\Theta}}(\textsf{y} \mid \boldsymbol{\mathsf{x}}) \to 1$ $\implies$ $\ell \to 0$
- $p_{\boldsymbol{\Theta}}(\textsf{y} \mid \boldsymbol{\mathsf{x}}) \to 0$ $\implies$ $\ell \to \infty$ 

Using an expectation over the underlying distribution allows the model to focus on errors based on its probability of occuring. For every set of parameters $\boldsymbol{\Theta},$ we approximate the **true risk** which is the expectation of $\ell$ on the underlying distribution with the **empirical risk** calculated on the sample $\mathcal{D}$:

$$
\begin{aligned}
\mathcal{L}(\boldsymbol{\Theta}) 
&= \mathbb{E}_{\boldsymbol{\mathsf{x}},y}\left[\ell(\textsf{y}, f_{\boldsymbol{\Theta}}(\boldsymbol{\mathsf{x}}))\right] \\
&\approx \frac{1}{|\mathcal{D}|} \sum_i \ell(\textsf{y}_i, f_{\boldsymbol{\Theta}}(\boldsymbol{\mathsf{x}}_i)) = \mathcal{L}_\mathcal{D}(\boldsymbol{\Theta}).
\end{aligned}
$$

The optimization problem can be written more generally as
$\hat{\boldsymbol{\Theta}} = \underset{\boldsymbol{\Theta}}{\text{argmin}}\, \mathcal{L}_\mathcal{D}(\boldsymbol{\Theta})
$.

### Cross-entropy

Note that the same input ${\boldsymbol{\mathsf{x}}}$ can have multiple labels in the dataset.
Consider the contribution $\mathcal{L}_{\boldsymbol{\mathsf{x}}}$ to the loss of the model's predictions $\hat{{p}}_{\boldsymbol{\mathsf{x}}} \in [0, 1]^C$ on an input $\boldsymbol{\mathsf{x}}.$ Suppose each label has occured $n^1, \ldots, n^C$ times given input ${\boldsymbol{\mathsf{x}}}$ out of $N$ input-output pairs in $\mathcal{D}.$ Let $n = n^1 + \ldots, n^C.$ Then, 

$$
\begin{aligned}
\mathcal{L}_{\boldsymbol{\mathsf{x}}} 
&= -\frac{1}{N} \, \Big(n^1 \log \hat{{p}}_{\boldsymbol{\mathsf{x}}}^1 + \ldots + n^C \log \hat{{p}}_{\boldsymbol{\mathsf{x}}}^C \Big)\\
&= \frac{1}{N} \, \left[n^1, \ldots, n^C \right] \cdot -\log \hat{{p}}_{\boldsymbol{\mathsf{x}}} \\
&= \frac{n}{N} \, {\left[\frac{n^1}{n}, \ldots, \frac{n^C}{n}\right]} \cdot -\log \hat{{p}}_{\boldsymbol{\mathsf{x}}}.
\end{aligned}
$$

Note that the dot product is the [cross-entropy](https://en.wikipedia.org/wiki/Cross-entropy#Cross-entropy_loss_function_and_logistic_regression) between[^crossent] model predict probabilities and the label distribution[^labeldist] given input $\boldsymbol{\mathsf{x}}.$ Finally, this cross-entropy is weighted by the empirical probability of $\boldsymbol{\mathsf{x}}$ occuring.
It follows that the NLL is equivalent to the *expected* cross-entropy between the model predict probabilities and the label distribution given an input. Consequently, any classification model trained to minimize cross-entropy on hard labels maximizes the likelihood of the training dataset.

[^crossent]: From [Gibbs' inequality](https://en.wikipedia.org/wiki/Gibbs%27_inequality), we have $H(p, q) \geq H(p, p)$. Hence, the cross-entropy is minimized when the model predict probabilities precisely match the label distribution. In principle, the cross entropy measures the amount of "information" needed to describe outputs of the model. Recall that the input output pairs $(\boldsymbol{\mathsf{x}}, y)$ are generated by a random process. The cross entropy increases as more information is needed to describe each outcome of this random process.

[^labeldist]: The empirical label distribution becomes a one-hot vector whenever $n = 1.$ In fact, we can convert each instance to a one-hot vector and calculate the loss as the expected cross-entropy with one-hot vectors as label distribution, even when an instance has multiple labels in the dataset, and get the same result. But deriving the probability distribution of the next state is important to understand as it occurs in a lot of scenarios (e.g. language modeling and RL).

**Example.** The PyTorch implementation of [`F.cross_entropy`](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) converts logits to probabilities using the softmax. Consistent with the above discussion, we can either pass hard labels $(B,)$ for a batch of $B$ inputs, or $(B, C)$ where $p_{ij} \in [0, 1]$ containing probabilities for class $j$ (soft labels) given instance $i.$

In [7]:
import torch
import torch.nn.functional as F

s = torch.tensor([
    [0.3333, 0.3333, 0.3333],
    [0.3333, 0.3333, 0.3333],
    [0.3333, 0.3333, 0.3333],
    [0.4333, 0.2333, 0.3333],
    [0.3333, 0.2333, 0.4333],
    [0.1333, 0.3333, 0.5333],
])
y = torch.tensor([0, 1, 1, 0, 1, 2])
F.cross_entropy(s, target=y)         # expects logits -> applies softmax

tensor(1.0686)

`F.cross_entropy` calculates cross-entropy with softmax probas:

In [8]:
q = F.softmax(s, dim=1)
-torch.log(q[range(s.shape[0]), y]).mean()

tensor(1.0686)

Following the above discussion, we can also use soft labels based on empirical label distribution:

In [9]:
p = torch.tensor([
    [0.3333, 0.6666, 0.0000],
    [0.3333, 0.6666, 0.0000],
    [0.3333, 0.6666, 0.0000],
    [1.0000, 0.0000, 0.0000],
    [0.0000, 1.0000, 0.0000],
    [0.0000, 0.0000, 1.0000]
])

F.cross_entropy(s, target=p)

tensor(1.0685)

Or with one-hot probability vectors:

In [10]:
p = torch.tensor([
    [1.0000, 0.0000, 0.0000],
    [0.0000, 1.0000, 0.0000],
    [0.0000, 1.0000, 0.0000],
    [1.0000, 0.0000, 0.0000],
    [0.0000, 1.0000, 0.0000],
    [0.0000, 0.0000, 1.0000]
])

F.cross_entropy(s, target=p)

tensor(1.0686)

## Gradient descent

Now that we have the empirical risk $\mathcal{L}_\mathcal{D}(\boldsymbol{\Theta})$ as objective, we proceed to the actual optimization algorithm. Given the current parameters $\boldsymbol{\Theta} \in \mathbb R^M$ of the network, we can imagine the network to be sitting on a point $(\boldsymbol{\Theta}, \mathcal L_{\mathcal{D}}(\boldsymbol{\Theta}))$ on a surface in $\mathbb R^M \times \mathbb R.$ The surface will generally vary for different samples of the training data. Training is equivalent to finding the minimum of this surface. 
Gradients arise in deep learning when making the following first-order approximation:

$$\Delta \mathcal L_{\mathcal{D}} \approx  \sum_k \left(\frac{\partial \mathcal L_{\mathcal{D}}}{ \partial {\Theta}_k} \right)  \Delta {\Theta}_k = \left( \nabla_{\boldsymbol{\Theta}}\, \mathcal L_{\mathcal{D}} \right) \cdot \Delta {\boldsymbol{\Theta}}.$$ 

It follows that $-\nabla_{\boldsymbol{\Theta}}\, \mathcal L_{\mathcal{D}}$ is the direction of steepest descent at the current point $(\boldsymbol{\Theta}, \mathcal L_{\mathcal{D}})$ in the surface. **Gradient descent** (GD) is defined by the update rule: 

$$
\boldsymbol{\Theta}^{t+1} = \boldsymbol{\Theta}^t - \eta\; \nabla_{\boldsymbol{\Theta}}\, \mathcal L_{\mathcal{D}}(\boldsymbol{\Theta}^t)
$$

where the **learning rate** $\eta > 0$ controls the step size. Note that finding a good set of **initial weights** $\boldsymbol{\Theta}^0 \in \mathbb{R}^M$ is crucial since networks has lots of internal symmetries that can cause it to diverge in the early stages of training. Later on we will see that other optimization algorithms in deep learning practice are just modifications of GD.

In [11]:
import numpy as np

def loss(w, X, y):
    return ((X @ w - y)**2).mean()

def grad(w, X, y, B=None):
    """Gradient step for the MSE loss function"""
    dw = 2*((X @ w - y).reshape(-1, 1) * X).mean(axis=0)
    return dw / np.linalg.norm(dw)

def grad_descent(w0, X, y, eta=0.1, steps=10):
    """Return sequence of weights from GD."""
    w = np.zeros([steps, 2])
    w[0, :] = w0
    for j in range(1, steps):
        u = w[j-1, :]
        w[j, :] = u - eta * grad(u, X, y)
    return w


# Generate data
n = 1000
X = np.zeros((n, 2))
X[:, 1] = np.random.uniform(low=-1, high=1, size=n)
X[:, 0] = 1
w_min = np.array([-1, 3])
y = (X @ w_min) + 0.05 * np.random.randn(n)  # data: y = -1 + 3x + noise

# Gradient descent
w_init = [-4, -4]
w_step = grad_descent(w_init, X, y, eta=1.0, steps=50)

In [12]:
from mpl_toolkits.mplot3d import Axes3D  # For 3D plotting

CMAP = "coolwarm"

def plot_surface(ax, data, target, N=50):
    x = np.linspace(-5, 5, N)
    y = np.linspace(-5, 5, N)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    for i in range(N):
        for j in range(N):
            Z[i, j] = loss(np.array([X[i, j], Y[i, j]]), data, target)
    
    ax.plot_surface(X, Y, Z, cmap=CMAP)
    ax.set_title("Loss surface")
    ax.set_xlabel(f'$w_0$')
    ax.set_ylabel(f'$w_1$')
    

def plot_contourf(ax, data, target, w_min, w_hist, N=50):
    x = np.linspace(-5, 5, N)
    y = np.linspace(-5, 5, N)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    for i in range(N):
        for j in range(N):
            Z[i, j] = loss(np.array([X[i, j], Y[i, j]]), data, target)

    for t in range(1, len(w_hist)):
        ax.annotate(
            "",
            xy=(w_hist[t][0], w_hist[t][1]), 
            xytext=(w_hist[t-1][0], w_hist[t-1][1]),
            arrowprops=dict(arrowstyle="->", color="k", linewidth=1)
        )


    c = ax.contourf(X, Y, Z, levels=20, cmap=CMAP)
    ax.scatter(w_min[0], w_min[1], color="yellow", s=30, label="min.", zorder=1)
    ax.scatter(w_hist[:, 0], w_hist[:, 1], marker="o", s=5, facecolors="k")
    ax.set_title("Gradient descent steps")
    ax.set_xlabel(f"$w_0$")
    ax.set_ylabel(f"$w_1$")
    ax.legend(loc="upper left", fontsize=8)
    plt.colorbar(c, ax=ax)


# Create a figure and two subplots
fig = plt.figure(figsize=(9, 3))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122)

# Call the functions with the respective axes
plot_surface(ax1, X, y)
plot_contourf(ax2, X, y, w_min, w_step)

plt.savefig("plots/00-gradient-descent.svg", bbox_inches="tight")
plt.close(fig);

```{figure} ./plots/00-gradient-descent.svg
---
align: center
label: plots/00-gradient-descent
width: 80%
---
**Visualizing the loss surface and gradient descent.** Shows linear function with MSE loss.
```

## Non-linear decision boundary

Going back to classification. Let us generate data that is not linearly separable. 
In this section, we will show that linear classification extends to input data that is not linearly separable. The idea is to apply a sequence of transformations (nonlinear, i.e. using activations) on the input $\boldsymbol{\mathsf{x}}$ such that the final features $f(\boldsymbol{\mathsf{x}})$ become linearly separable.

In [13]:
import torch 
torch.manual_seed(2)

N = 1500  # sample size
noise = lambda e: torch.randn(N, 2) * e
t = 2 * torch.pi * torch.rand(N, 1)
s = 2 * torch.pi * torch.rand(N, 1)

x0 = torch.cat([0.1 * torch.cos(s), 0.1 * torch.sin(s)], dim=1) + noise(0.05)
x1 = torch.cat([1.0 * torch.cos(t), 1.0 * torch.sin(t)], dim=1) + noise(0.1)
y0 = (torch.ones(N,) * 0).long()
y1 = (torch.ones(N,) * 1).long()

In [14]:
%config InlineBackend.figure_format = "svg"
import matplotlib.pyplot as plt

plt.scatter(x0[:, 0], x0[:, 1], s=10.0, edgecolor="k", label=0, color="C0")
plt.scatter(x1[:, 0], x1[:, 1], s=10.0, edgecolor="k", label=1, color="C1")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.legend()
plt.axis("equal")

plt.savefig("plots/00-circles-data.svg", bbox_inches="tight")
plt.close();

```{figure} ./plots/00-circles-data.svg
---
align: center
label: plots/00-circles-data
width: 70%
---
**Concentric circles.** This toy dataset will be used as a benchmark for a basic neural network.
```

Modeling this with a FNN with one hidden layer containing units that uses the [tanh function](https://mathworld.wolfram.com/HyperbolicTangent.html) as activation: $\text{tanh}(z) = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}.$
This maps $\mathbb{R}$ to $[-1, 1]$ symmetrically with $\text{tanh}(0) = 0$ and $\text{tanh}(z) \to \pm 1$ as $z \to \pm\infty.$ Note that tanh is actually just a scaled and translated version of the sigmoid function.

In [15]:
import torch.nn as nn
from torchsummary import summary

model = lambda: nn.Sequential(
    nn.Linear(2, 3), nn.Tanh(),
    nn.Linear(3, 2)
)

summary(model(), input_size=(2,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                    [-1, 3]               9
              Tanh-2                    [-1, 3]               0
            Linear-3                    [-1, 2]               8
Total params: 17
Trainable params: 17
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


As discussed, we perform gradient descent on the cross-entropy loss:

In [16]:
import torch.nn.functional as F
from tqdm import tqdm

net = model()
optim = torch.optim.SGD(net.parameters(), lr=0.01)

x = torch.cat([x0, x1])
y = torch.cat([y0, y1])
history = {"accs": [], "loss": []}
for step in tqdm(range(25000)):
    s = net(x)
    loss = F.cross_entropy(s, y)
    loss.backward()
    optim.step()
    optim.zero_grad()
    history["loss"].append(loss.item())
    history["accs"].append(100 * (y == torch.argmax(s, dim=1)).float().mean())

100%|██████████| 25000/25000 [00:18<00:00, 1383.97it/s]


In [17]:
fig, ax1 = plt.subplots(figsize=(8, 3))
ax2 = ax1.twinx()

ax1.plot(history["loss"], color="blue", linewidth=2)
ax2.plot(history["accs"], color="red",  linewidth=2)
ax1.set_xlabel("step")
ax1.ticklabel_format(axis="x", style="sci", scilimits=(3, 3))
ax1.grid(axis="both", linestyle="dotted", alpha=0.8)

ax1.set_ylabel("Batch loss")
ax2.set_ylabel("Batch accs (%)")
ax1.yaxis.label.set_color("blue")
ax2.yaxis.label.set_color("red")

plt.savefig("plots/00-circles-train.svg", bbox_inches="tight")
plt.close(fig);

```{figure} ./plots/00-circles-train.svg
---
align: center
label: plots/00-circles-train
---
**Visualizing the training process.** Observe that accuracy trend does not exactly match the decreasing loss. This is expected since accuracy considers hard labels whereas the loss is calculated with respect to soft probability distributions.
```

In [18]:
import warnings
warnings.filterwarnings("ignore")

# transformations
with torch.no_grad():
    linear_0 = net[0](x0)
    linear_1 = net[0](x1)
    linear_act_0 = net[1](net[0](x0))
    linear_act_1 = net[1](net[0](x1))

    # separating hyperplane (see above discussion, i.e. w <- w1 - w0  == logistic reg)
    h = 1
    w, b = net[2].parameters()
    w, b = (w[h] - w[h-1]), (b[h] - b[h-1])

# plot
fig = plt.figure(figsize=(12, 4))
ax0 = fig.add_subplot(131)
ax1 = fig.add_subplot(132, projection='3d')
ax2 = fig.add_subplot(133, projection='3d')

ax0.grid(alpha=0.8, linestyle="dotted")
ax0.set_axisbelow(True)
ax0.scatter(x0[:, 0], x0[:, 1], s=2.0, label=0, color="C0")
ax0.scatter(x1[:, 0], x1[:, 1], s=2.0, label=1, color="C1")
ax0.set_xlabel("$x_1$")
ax0.set_ylabel("$x_2$")
ax0.set_xlim(-1.5, 1.5)
ax0.set_ylim(-1.5, 1.5)
ax0.set_title("(a) input")
ax0.legend()
ax0.axis('equal')

ax1.scatter(linear_0[:, 0], linear_0[:, 1], linear_0[:, 2], s=3, label=0, color="C0")
ax1.scatter(linear_1[:, 0], linear_1[:, 1], linear_1[:, 2], s=3, label=1, color="C1")
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
ax1.set_zlabel('$x_3$')
ax1.set_title('(b) linear')

ax2.scatter(linear_act_0[:, 0], linear_act_0[:, 1], linear_act_0[:, 2], s=3, label=0, color="C0")
ax2.scatter(linear_act_1[:, 0], linear_act_1[:, 1], linear_act_1[:, 2], s=3, label=1, color="C1")
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_zlabel('$x_3$')
ax2.set_title('(c) linear + tanh')

# Generate grid of points
x_min = min(linear_act_1[:, 0].min(), linear_act_0[:, 0].min())
x_max = max(linear_act_1[:, 0].max(), linear_act_0[:, 0].max())
y_min = min(linear_act_1[:, 1].min(), linear_act_0[:, 1].min())
y_max = max(linear_act_1[:, 1].max(), linear_act_0[:, 1].max())
a, b, c, d = w[0], w[1], w[2], b
x = np.linspace(x_min, x_max, 50)
y = np.linspace(y_min, y_max, 50)
X, Y = np.meshgrid(x, y)
Z = (-a * X - b * Y - d) / c

# Plot the hyperplane for the positive class
ax2.plot_surface(X, Y, Z, alpha=0.5, color=f"C{h}")
fig.tight_layout()
plt.savefig("plots/00-circles-transformation.svg", bbox_inches="tight")
plt.close(fig);

```{figure} ./plots/00-circles-transformation.svg
---
align: center
label: plots/00-circles-transformation
---
**Feature transformation at each layer.** Feature representations are iteratively transformed. The final features are separated by a hyperplane. Here the two weight vectors fuse resulting in one separating plane. In general, we need $K - 1$ hyperplanes for $K$ classes.
```

Note that we can project the decision hyperplane back to the input space:

In [19]:
from matplotlib.colors import LinearSegmentedColormap

# define custom colormap
colors = ["C0", "C1"]
n_bins = 100
cm = LinearSegmentedColormap.from_list(name="", colors=colors, N=n_bins)

# create a grid of points
N = 100
x = np.linspace(-1.8, 1.8, N)
y = np.linspace(-1.8, 1.8, N)
X, Y = np.meshgrid(x, y)

# calculate p1 for each point in grid
Z = np.zeros_like(X)
for i in range(N):
    for j in range(N):
        out = F.softmax(net(torch.tensor([[float(X[i, j]), float(Y[i, j])]])), dim=1)
        Z[i, j] = out[0][1]

# create a color plot
plt.pcolormesh(X, Y, Z, shading="auto", cmap=cm, rasterized=True)
plt.colorbar()
plt.xlabel("X")
plt.ylabel("Y")

plt.scatter(x0[:, 0], x0[:, 1], s=10.0, label=0, color="C0", edgecolor="black")
plt.scatter(x1[:, 0], x1[:, 1], s=10.0, label=1, color="C1", edgecolor="black")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.legend()
plt.axis("equal")

plt.ylim(-1.5, 1.5)
plt.savefig("plots/00-circles-decision.svg", bbox_inches="tight")
plt.close("all");

Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


```{figure} ./plots/00-circles-decision.svg
---
align: center
label: plots/00-circles-decision
---
**Projecting the linear decision surface on the input space.** The pre-image is obtained by simply calculating the probability assigned by the model to each data point in the input space. Using ReLU activation here instead of Tanh results in a boundary with sharp corners. This can be thought of as a manifestation of inductive bias.
```

Checking classification accuracy:

In [20]:
a = (torch.argmax(net(x0), dim=1) == y0).float().mean().item()
b = (torch.argmax(net(x1), dim=1) == y1).float().mean().item()
a, b

(1.0, 1.0)

---